# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 1


In [ ]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"


In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# ============================================================
# Environment Configuration
# ============================================================

# Required environment variables for the Udaplay RAG Agent
REQUIRED_ENV_VARS = (
    "OPENAI_API_KEY",
    "OPENAI_BASE_URL",
    "TAVILY_API_KEY",
)

# Locate .env in the current directory or parent directories
env_path = find_dotenv(usecwd=True)

# Load configuration without overwriting existing variables
if env_path:
    load_dotenv(dotenv_path=env_path, override=False)
    print(f"✓ Environment file found: {Path(env_path).parent}")
else:
    print("ℹ No .env file found. Checking system environment variables.")

# Retrieve and validate required configuration
config = {
    name: (os.getenv(name) or "").strip()
    for name in REQUIRED_ENV_VARS
}

# Identify missing or empty environment variables
missing_vars = [
    name for name, value in config.items()
    if not value
]

if missing_vars:
    raise EnvironmentError(
        "Missing required environment variables: "
        + ", ".join(missing_vars)
        + "\nCheck your .env file or system environment configuration."
    )

# Confirm successful validation without exposing credentials
print("\n✓ Environment configuration validated successfully!")

for name in REQUIRED_ENV_VARS:
    print(f"✓ {name}: Configured")


✓ Environment file found: /workspace/Code/project/starter

✓ Environment configuration validated successfully!
✓ OPENAI_API_KEY: Configured
✓ OPENAI_BASE_URL: Configured
✓ TAVILY_API_KEY: Configured


### VectorDB Instance

In [15]:

import os
from pathlib import Path

import chromadb
from chromadb.config import Settings

# ============================================================
# ChromaDB Persistent Client Configuration
# ============================================================

# Resolve the database directory to an absolute path.
# CHROMA_DB_PATH can optionally be configured in the .env file.
DEFAULT_DB_PATH = Path.cwd() / "chromadb"

CHROMA_DB_PATH = Path(
    os.getenv("CHROMA_DB_PATH", str(DEFAULT_DB_PATH))
).expanduser().resolve()

# Configure ChromaDB
chroma_settings = Settings(
    anonymized_telemetry=False,  # Disable anonymized telemetry
    allow_reset=False,           # Protect existing data
)

# ============================================================
# Initialize and Validate ChromaDB
# ============================================================

try:
    # Ensure the storage directory exists
    CHROMA_DB_PATH.mkdir(parents=True, exist_ok=True)

    # Create or reconnect to the persistent vector database
    chroma_client = chromadb.PersistentClient(
        path=str(CHROMA_DB_PATH),
        settings=chroma_settings,
    )

    # Verify that ChromaDB responds successfully
    chroma_client.heartbeat()

    # Retrieve existing collection count
    collection_count = chroma_client.count_collections()

except Exception as exc:
    raise RuntimeError(
        f"Failed to initialize ChromaDB at '{CHROMA_DB_PATH}'. "
        f"Reason: {exc}"
    ) from exc

# ============================================================
# Initialization Summary
# ============================================================

print("\n✓ ChromaDB initialized successfully!")
print(f"✓ ChromaDB Version: {chromadb.__version__}")
print(f"✓ Database Path: {CHROMA_DB_PATH}")
print(f"✓ Persistence: Enabled")
print(f"✓ Telemetry: Disabled")
print(f"✓ Existing Collections: {collection_count}")
print("✓ Database Ready for RAG Operations")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given



✓ ChromaDB initialized successfully!
✓ ChromaDB Version: 1.0.4
✓ Database Path: /workspace/Code/project/starter/chromadb
✓ Persistence: Enabled
✓ Telemetry: Disabled
✓ Existing Collections: 2
✓ Database Ready for RAG Operations


### Collection

In [18]:

import os
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# ============================================================
# OpenAI Embedding Function Configuration
# ============================================================

# Select the embedding model
EMBEDDING_MODEL = os.getenv(
    "OPENAI_EMBEDDING_MODEL",
    "text-embedding-3-small"
)

# Retrieve environment configuration
OPENAI_API_KEY = (os.getenv("OPENAI_API_KEY") or "").strip()
OPENAI_BASE_URL = (os.getenv("OPENAI_BASE_URL") or "").strip()

# ============================================================
# Validate Configuration
# ============================================================

if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is missing. "
        "Verify that your .env file has been loaded."
    )

# ============================================================
# Initialize Embedding Function
# ============================================================

try:
    embedding_fn = OpenAIEmbeddingFunction(
        model_name=EMBEDDING_MODEL,
        api_key_env_var="OPENAI_API_KEY",
        api_base=OPENAI_BASE_URL or None
    )

except Exception as exc:
    raise RuntimeError(
        f"Failed to initialize embedding model "
        f"'{EMBEDDING_MODEL}': {exc}"
    ) from exc

# ============================================================
# Initialization Summary
# ============================================================

print("\n✓ OpenAI Embedding Function Initialized")
print(f"✓ Model: {EMBEDDING_MODEL}")
print("✓ API Key: Configured")
print(
    f"✓ API Endpoint: "
    f"{'Custom endpoint' if OPENAI_BASE_URL else 'OpenAI default'}"
)
print("✓ Ready for ChromaDB vector operations")



✓ OpenAI Embedding Function Initialized
✓ Model: text-embedding-3-small
✓ API Key: Configured
✓ API Endpoint: Custom endpoint
✓ Ready for ChromaDB vector operations


In [19]:

# ============================================================
# ChromaDB Collection Configuration
# ============================================================

COLLECTION_NAME = "udaplay_games"
EMBEDDING_MODEL = "text-embedding-3-small"
DISTANCE_METRIC = "cosine"

# Set True only when intentionally rebuilding the collection.
# WARNING: This deletes all existing documents and embeddings.
RESET_COLLECTION = False

# Descriptive information about the collection
COLLECTION_METADATA = {
    "description": "Video game knowledge base for UdaPlay RAG agents",
    "domain": "video_games",
    "embedding_model": EMBEDDING_MODEL,
}

# Vector index configuration (ChromaDB 1.x)
COLLECTION_CONFIGURATION = {
    "hnsw": {
        "space": DISTANCE_METRIC,
    }
}

# ============================================================
# Collection Initialization
# ============================================================

try:
    # Retrieve existing collection names
    existing_collections = {
        getattr(item, "name", item)
        for item in chroma_client.list_collections()
    }

    collection_exists = COLLECTION_NAME in existing_collections

    # Optionally delete an existing collection
    if RESET_COLLECTION and collection_exists:
        chroma_client.delete_collection(name=COLLECTION_NAME)
        collection_exists = False

        print(f"✓ Existing collection '{COLLECTION_NAME}' deleted")

    # Create the collection or reconnect to the existing one
    collection = chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata=COLLECTION_METADATA,
        configuration=COLLECTION_CONFIGURATION,
    )

    # Validate the distance metric when configuration is available
    config = getattr(collection, "configuration", None) or {}

    if isinstance(config, dict):
        existing_metric = (config.get("hnsw") or {}).get("space")

        if existing_metric and existing_metric != DISTANCE_METRIC:
            raise ValueError(
                f"Existing collection uses '{existing_metric}', "
                f"but '{DISTANCE_METRIC}' was requested. "
                "Use a different collection name or explicitly rebuild it."
            )

    document_count = collection.count()

except Exception as exc:
    raise RuntimeError(
        f"Failed to initialize ChromaDB collection "
        f"'{COLLECTION_NAME}': {exc}"
    ) from exc

# ============================================================
# Collection Initialization Summary
# ============================================================

print("\n✓ ChromaDB Collection Ready")
print(f"✓ Collection Name: {collection.name}")
print(f"✓ Status: {'Reconnected' if collection_exists else 'Created'}")
print(f"✓ Embedding Model: {EMBEDDING_MODEL}")
print(f"✓ Distance Metric: {DISTANCE_METRIC}")
print(f"✓ Stored Documents: {document_count}")
print("✓ Persistent Storage: Enabled")
print("✓ Ready for Video Game Data Ingestion")


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



✓ ChromaDB Collection Ready
✓ Collection Name: udaplay_games
✓ Status: Reconnected
✓ Embedding Model: text-embedding-3-small
✓ Distance Metric: cosine
✓ Stored Documents: 15
✓ Persistent Storage: Enabled
✓ Ready for Video Game Data Ingestion


### Add documents

In [20]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Test a Semantic Search Function

In [21]:

# ============================================================
# UdaPlay RAG - Semantic Search Validation
# ============================================================

TEST_QUERIES = [
    "racing games from the 90s",
    "RPG games with turn-based combat",
    "action games for PlayStation",
]

MAX_RESULTS = 3
DESCRIPTION_LENGTH = 150

# ============================================================
# Helper: Safely Format Text
# ============================================================

def truncate_text(text: str, max_length: int = 150) -> str:
    """Normalize whitespace and truncate long descriptions."""
    if not text:
        return "Description unavailable"

    text = " ".join(str(text).split())

    if len(text) <= max_length:
        return text

    return text[:max_length - 3].rstrip() + "..."


# ============================================================
# Validate Vector Database
# ============================================================

try:
    total_documents = collection.count()
except Exception as exc:
    raise RuntimeError(
        f"Unable to access ChromaDB collection: {exc}"
    ) from exc

if total_documents == 0:
    raise ValueError(
        "The ChromaDB collection is empty. "
        "Load and embed the video game dataset before searching."
    )

print("=" * 80)
print("UDAPLAY - SEMANTIC SEARCH VALIDATION")
print("=" * 80)
print(f"Collection: {collection.name}")
print(f"Indexed Documents: {total_documents}")
print(f"Test Queries: {len(TEST_QUERIES)}")
print(f"Maximum Results per Query: {MAX_RESULTS}")

# ============================================================
# Execute Semantic Search Queries
# ============================================================

successful_queries = 0

for query_number, query in enumerate(TEST_QUERIES, start=1):

    print(f"\n{'=' * 80}")
    print(f"QUERY {query_number}: {query}")
    print("-" * 80)

    try:
        results = collection.query(
            query_texts=[query],
            n_results=min(MAX_RESULTS, total_documents),
            include=["metadatas", "documents", "distances"],
        )

        # ChromaDB returns nested lists for query results
        ids = (results.get("ids") or [[]])[0]
        metadatas = (results.get("metadatas") or [[]])[0]
        documents = (results.get("documents") or [[]])[0]
        distances = (results.get("distances") or [[]])[0]

        if not ids:
            print("No matching documents returned.")
            continue

        print(f"Retrieved {len(ids)} candidate matches")

        # Display ranked search results
        for index, game_id in enumerate(ids):

            metadata = (
                metadatas[index]
                if index < len(metadatas)
                else None
            ) or {}

            document = (
                documents[index]
                if index < len(documents)
                else None
            )

            distance = (
                distances[index]
                if index < len(distances)
                else None
            )

            # Extract game attributes safely
            game_name = metadata.get("Name", "Unknown Game")
            release_year = metadata.get("YearOfRelease", "Unknown")
            platform = metadata.get("Platform", "Unknown")

            # Prefer metadata description; fallback to document
            description = (
                metadata.get("Description")
                or document
                or ""
            )

            print(f"\n{index + 1}. {game_name} ({release_year})")
            print(f"   Platform: {platform}")
            print(f"   Record ID: {game_id}")

            if distance is not None:
                print(f"   Cosine Distance: {distance:.4f}")

            print(
                "   Description: "
                + truncate_text(description, DESCRIPTION_LENGTH)
            )

        successful_queries += 1

    except Exception as exc:
        print(f"Search failed for '{query}': {exc}")

# ============================================================
# Execution Summary
# ============================================================

print(f"\n{'=' * 80}")
print("SEMANTIC SEARCH VALIDATION SUMMARY")
print("=" * 80)
print(f"Queries Executed: {len(TEST_QUERIES)}")
print(f"Queries Returning Results: {successful_queries}")
print(
    f"Queries Without Results or Failed: "
    f"{len(TEST_QUERIES) - successful_queries}"
)

if successful_queries == len(TEST_QUERIES):
    print("\n✓ All semantic search queries returned results!")
else:
    print("\n⚠ Review queries without results or with errors.")


UDAPLAY - SEMANTIC SEARCH VALIDATION
Collection: udaplay_games
Indexed Documents: 15
Test Queries: 3
Maximum Results per Query: 3

QUERY 1: racing games from the 90s
--------------------------------------------------------------------------------


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 3 candidate matches

1. Gran Turismo (1997)
   Platform: PlayStation 1
   Record ID: 001
   Cosine Distance: 0.4355
   Description: A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

2. Gran Turismo 5 (2010)
   Platform: PlayStation 3
   Record ID: 003
   Cosine Distance: 0.5530
   Description: A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

3. Pokémon Ruby and Sapphire (2002)
   Platform: Game Boy Advance
   Record ID: 007
   Cosine Distance: 0.6554
   Description: Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.

QUERY 2: RPG games with turn-based combat
--------------------------------------------------------------------------------
Retrieved 3 candidate matches

1. Pokémon Ruby and Sapphire (2002)
   Platform: Game Boy Advance
   Record ID: 007
   Cosine Distance: 0.6733
   Description: Third-gene